# **DHSI 2026 — Computational Text Analysis**
## **Day Two: Introduction to Stylometry**

# **General Statistics**
Here we will explore some basic metrics about text statistics. Not really the state of the art, but it's a good place to start.

In [ ]:
import os
import spacy
import re
import sys
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from prettytable import PrettyTable
from tqdm import tqdm
import sys
sys.path.append ('..')
from progress.checkpoint import checkpoint
from toolbox_gs import corpus_stats, author_stats, plot_authors, plot_authors_3d, word_frequency, word_frequency_plot, word_frequency_combined, top_words_plot, compare_authors_heatmap, cumulative_coverage_plot,sentence_length_histogram, word_length_histogram, sentence_length_over_text, word_length_over_text, zipf_plot, ttr_curve, punctuation_density

SESSION_ID = 'General Statistics'
checkpoint (SESSION_ID, 'imports')

In [ ]:
#############################################################
#                                                           #
#   CONFIGURATION (EDIT THIS CELL TO CHANGE SETTINGS)       #
#                                                           #
#############################################################

input_folder_forms = "/Users/jacek_bakowski/DHSI_2026_CTA/corpora/A_Small_Collection_of_British_Fiction/raw/corpus"
input_folder_lemmas = "/Users/jacek_bakowski/DHSI_2026_CTA/corpora/A_Small_Collection_of_British_Fiction/lemmatized/corpus"

try:
    assert os.path.exists (input_folder_forms)
    assert os.path.exists (input_folder_lemmas)
    checkpoint (SESSION_ID, 'data')
except AssertionError:
    if not os.path.exists (input_folder_forms):
        print (f'The path {input_folder_forms} doesn\'t exist, please correct it')
    if not os.path.exists (input_folder_lemmas):
        print (f'The path {input_folder_lemmas} doesn\'t exist, please correct it')

## **Stats per File**

<div style="
  border-left: 6px solid #3B82F6;
  background: rgba(127,127,127,0.08);
  padding: 16px 20px;
  margin: 18px 0;
  border-radius: 10px;
">

<b>Chars</b> -> number og characters (excluding whitespace) per text file + aggregate for the whole corpus

<b>Words</b> -> number of word tokens per text file

<b>Types</b> -> number of unique word types per text file

<b>Sentences</b> -> number of sentences per text file

<b>Avg word</b> -> average word length per text file

<b>Avg sent</b> -> average sentence length in words per text file

<b>TTR-@n</b> -> TTR averaged overnon-overlapping windows of n tokens per text file

<b>Hapax %</b> -> percentage of word types that appear exactly once per text file : number of hapax legomena / number of word types * 100

<b>Hapax-@n</b> -> Hapax types as a percentage of all types, averaged across non-overlapping windows of n tokens
</div>

In [ ]:
# Just change the value of the variable input_folder to analyze a different corpus
# Change the value of sample_size to analyze a smaller or larger sample of texts
corpus_stats(input_folder=input_folder_forms, sample_size=5000)

## Authors stats

In [ ]:
author_stats(input_folder_forms, sample_size=5000)

## Authors plot

In [ ]:
# for x and y_metric choose one of those:
# avg_book_len, avg_word, avg_sent,ttr_n, hapax_pct, hapax_n

fig = plot_authors(
    input_folder_forms,
    x_metric="avg_sent",
    y_metric="ttr_n",
    size_metric="words",
    sample_size=5000,
    title="Authors in avg_sent vs. Hapax % space (point size \u221d number of books)",
)

## 3D Authors plot

In [ ]:
# for x, y, and z_metric choose one of those:
# avg_book_len, avg_word, avg_sent, ttr_n, hapax_pct, hapax_n
input_folder_no_stopwords = "/Users/jacek_bakowski/DHSI_2026_CTA/corpora/A_Small_Collection_of_British_Fiction/raw_stop_words_removed/corpus"
anim = plot_authors_3d(
    input_folder_no_stopwords,
    x_metric="avg_sent",
    y_metric="ttr_n",
    z_metric="hapax_n",
    sample_size=5000
)
from IPython.display import HTML
HTML(anim.to_jshtml())

## Word frequencies

In [ ]:
word_frequency(input_folder_forms, ["love", "death", "god"])

## Stop-Words (Function Words)

In [ ]:
input_file = os.path.expanduser("/Users/jacek_bakowski/DHSI_2026_CTA/corpora/A_Small_Collection_of_British_Fiction/raw/corpus/Austen_Emma.txt")
fig = top_words_plot(input_file, n=30)

In [ ]:
input_file = os.path.expanduser("/Users/jacek_bakowski/DHSI_2026_CTA/corpora/A_Small_Collection_of_British_Fiction/raw/corpus/Austen_Emma.txt")
fig = cumulative_coverage_plot(input_file, max_n=500)

## Heatmaps of word frequencies

In [ ]:
fig = compare_authors_heatmap(input_folder_forms, n=30) 

In [ ]:
fig = compare_authors_heatmap(input_folder_forms, n=30, exclude_stop_words=True) 

# **Bonus Materials**
# **Beyond averages: distributions and trajectories**
Beware! Descriptive statistics like **"Average sentence length = 18 words"** are useful, but they hide enormous amounts of structure.

Two texts with identical mean sentence length can read completely differently — one alternates rapid dialogue and long descriptions, the other plods along uniformly. 

### ***Each plot adds one more observation about the same data!!!***

To see that we look at:

### ***Sentence Length Histogram:***

In [ ]:
input_file = input_folder_forms + "/Austen_Emma.txt"
fig = sentence_length_histogram(input_file)

In [ ]:
input_file = input_folder_forms + "/EBronte_Wuthering.txt"
fig = sentence_length_histogram(input_file)

In [ ]:
input_file = input_folder_forms + "/Richardson_Clarissa.txt"
fig = sentence_length_histogram(input_file)

### **Word length histogram**

In [ ]:
input_file = input_folder_forms + "/EBronte_Wuthering.txt"
fig = word_length_histogram(input_file)

In [ ]:
input_file = input_folder_forms + "/EHemingway_OldMan.txt"
fig = word_length_histogram(input_file)

In [ ]:
input_file = input_folder_forms + "/Richardson_Clarissa.txt"
fig = word_length_histogram(input_file)

# Trajectories (over the text)
- **`sentence_length_over_text(file, window=20)`** - rolling mean of sentence length accross the book. Peaks usually mark descriptive or contemplative passages; troughs mark dialogue or action.
- **`word_length_over_text(file, chunk_size=1000)`** - register shifts visible as up/down jumps. Useful for finding the boundary between, say, a scientific preface and a narrative.

## Reading the trajectory plots

Both `sentence_length_over_text` and `word_length_over_text` show how a measure changes **across the text**, from start to end. They help you see where in the book the style shifts — without reading it.

### `sentence_length_over_text(file, window=20)`

Each point on the line represents **one sentence**, with the y-value being the **mean length of `window` sentences centred on that point**.

- **X-axis**: sentence index (1, 2, 3, ...)
- **Y-axis**: rolling average of sentence length in words
- **One point per sentence**

With `window=20`, sentence number 50 shows the average of sentences 40–60. Smaller window = less smoothing, more local noise. Larger window = smoother curve, only big structural moves are visible. Set `window=1` if you want raw lengths with no smoothing.

**What to look for:** dips usually mark dialogue or rapid action ("Yes." "No." "Are you sure?"), peaks mark descriptive or contemplative passages with longer constructions. A flat line means uniform pacing.

### `word_length_over_text(file, chunk_size=1000)`

Each point represents one **non-overlapping chunk** of `chunk_size` tokens — not one sentence.

- **X-axis**: token position in the text (the chunk's centre)
- **Y-axis**: mean word length (characters) in that chunk
- **Number of points**: `total_tokens / chunk_size`

For a 80 000-word novel with `chunk_size=1000`, you get about 80 points. Each is plotted at the **middle** of its chunk (the first at position 500, the next at 1500, and so on).

**What to look for:** rises indicate longer-word passages — technical or formal register, descriptive prose, scientific terminology. Drops indicate sections heavy in short words — dialogue, action, simple narration. The red dashed line is the overall mean for comparison.

### Why the two work differently

Sentences are natural discrete units, so a rolling window of *N sentences* is intuitive. Words are too granular for that — 80 000 rolling points per book would be unreadable — so we group words into bigger chunks and plot one point per chunk instead. The visual idea is the same: smooth a noisy signal so the underlying structure becomes visible.

### Choosing the parameter

| | Small | Medium (default) | Large |
|---|---|---|---|
| `sentence_length_over_text` window | `5` — see chapter-by-chapter | `20` | `50` — see book-wide trends only |
| `word_length_over_text` chunk_size | `200` — micro register shifts | `1000` | `5000` — coarse trends |

### A quick experiment

Try both on the same novel:

```python
sentence_length_over_text("given_file.txt", window=20)
word_length_over_text("data/given_file", chunk_size=1000)
```

You'll often see the two curves move together (a dialogue section has both short sentences *and* short words) but not always — descriptive prose can have long sentences full of short words, and a technical passage can have short sentences full of long words. The combination tells you more than either alone.


### **Sentence length over text**

In [ ]:
input_file = input_folder_forms + "/EBronte_Wuthering.txt"
fig = sentence_length_over_text(input_file, window=20)

In [ ]:
input_file = input_folder_forms + "/Richardson_Clarissa.txt"
fig = sentence_length_over_text(input_file, window=20)

### **Word length over text**

In [ ]:
input_file = input_folder_forms + "/EBronte_Wuthering.txt"
fig = word_length_over_text(input_file)

In [ ]:
input_file = input_folder_forms + "/Richardson_Clarissa.txt"
fig = word_length_over_text(input_file)

In [ ]:
fig = word_frequency_combined(input_folder_forms, ["love", "death", "god"], chunk_size=1000)

In [ ]:
fig = word_frequency_combined(input_folder_forms, ["love", "death", "god"], chunk_size=10000)

## How to lemmatize texts?
If you need to lemmatize your corpus, this can be helpful:

In [ ]:
def lemmatize_text(nlp, text, chunk_size=200000):
    # Process large texts in chunks to avoid spaCy limits
    parts = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        doc = nlp(chunk)

        parts.append("".join(
            (token.text if token.lemma_ == "-PRON-" else token.lemma_) + token.whitespace_
            for token in doc
        ))

    return "".join(parts)


def process_folder(input_dir, output_dir, extension=".txt"):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Load spaCy model (faster without parser/NER)
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

    # Collect all files first for progress bar
    all_files = []

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if extension and not file.endswith(extension):
                continue

            all_files.append((root, file))

    # Progress bar over all files
    for root, file in tqdm(all_files, desc="Lemmatizing corpus"):
        relative_path = os.path.relpath(root, input_dir)
        target_root = os.path.join(output_dir, relative_path)
        os.makedirs(target_root, exist_ok=True)

        input_file = os.path.join(root, file)
        output_file = os.path.join(target_root, f"{os.path.splitext(file)[0]}_lemmatized{os.path.splitext(file)[1]}")

        # Read file
        with open(input_file, "r", encoding="utf-8") as f:
            text = f.read()

        # Lemmatize
        lemmatized = lemmatize_text(nlp, text)

        # Write output
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(lemmatized)

In [ ]:
# And now let's lemmatize it!
### Beware: this will take a while, especially if you have a lot of files. You can interrupt it with Ctrl+C if needed.
### For the whole corpus it may take around 15 minutes, but it depends on your machine.

input_folder = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/")
output_folder = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus/")

process_folder(input_folder, output_folder)

# Bonus Session: Zipf's Law

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Austen_Emma.txt")
fig = zipf_plot(input_file)

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Austen_Emma.txt")
fig = ttr_curve(input_file)